In [39]:
model_id = 'sherlock'

In [38]:
%env PYTHONHASHSEED=13
%env PYTHONHASHSEED

env: PYTHONHASHSEED=13


'13'

In [1]:
from pandas.core.arrays.integer import dtype
%load_ext autoreload
%autoreload 2

In [35]:
import itertools

from ast import literal_eval
from collections import Counter
from datetime import datetime

import pandas as pd
import numpy as np

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
#from sklearn.ensemble import ExtraTreesClassifier,
#from sklearn.preprocessing import LabelEncoder
#from sklearn.metrics import classification_report, f1_score

In [3]:
#from sherlock.deploy.model import SherlockModel

In [4]:
LABEL_MAP = {
    # Date
    "Vaccination_date": "date",
    "Date_report":"date",
    "Date_onset":  "date",
    "Date_confirmation": "date",
    "Date_of_first_consultation":"date",
    "Date_hospitalisation":  "date",
    "Date_discharge_hospital": "date",
    "Date_admission_ICU":   "date",
    "Date_discharge_ICU":  "date",
    "Date_isolation":  "date",
    "Date_death":  "date",
    "Date_recovered":  "date",
    "Travel_history_entry": "date",
    "Travel_history_start":  "date",
    "Date_entry":  "date",
    "Date_last_modified": "date",

    # ID
    "Contact_ID": "id",
    "ID": "id",

    #Gender
    "Gender": "gender",
    "Sex_at_birth": "gender",
    "Gender_other": "gender",
    "Sex_at_birth_other": "gender",

    #Location
    "Travel_history_location": "location",
    "Location_information": "location",

    # Contact setting
    "Contact_setting": "contact_setting",
    "Contact_setting_other": "contact_setting",

    # demographic
    "Race": "demographic",
    "Ehtnicity": "demographic",

    # Medical Boolean
    "Healthcare_worker": "medical_boolean",
    "Previous_infection": "medical_boolean",
    "Pregnancy_Status": "medical_boolean",
    "Vaccination":  "medical_boolean",
    "Hospitalised":  "medical_boolean",
    "Intensive_care":  "medical_boolean",
    "Home_monitoring":  "medical_boolean",
    "Isolated": "medical_boolean",
    "Contact_with_case": "medical_boolean",
    "Travel_history": "medical_boolean",

    # Sourec
    "Source": "source",
    "Source_II": "source",
    "Source_III": "source",
    "Source_IV": "source",
}

In [5]:
LABEL_MAP_LC = {k.lower(): v.lower() for k, v in LABEL_MAP.items()}

def remap_labels(arr, mapping=LABEL_MAP_LC):
    """
    • forces each element to lower-case
    • replaces it if the key exists in `mapping`
    • otherwise leaves it as lower-case original
    """
    return np.array([mapping.get(x.lower(), x.lower()) for x in arr])

In [11]:
data_dir = "../custom_data"

### Train

In [22]:
X_train = pd.read_parquet(f"{data_dir}/processed/train.parquet")
y_train = pd.read_parquet(f"{data_dir}/raw/train_labels.parquet").values.flatten()
y_train = remap_labels(y_train)
y_train = np.array([x.lower() for x in y_train])

### Validation

In [23]:
X_validation = pd.read_parquet(f"{data_dir}/processed/validation.parquet")
y_validation = pd.read_parquet(f"{data_dir}/raw/validation_labels.parquet").values.flatten()
y_validation = remap_labels(y_validation)
y_validation = np.array([x.lower() for x in y_validation])

### Testing

In [24]:
X_test = pd.read_parquet(f"{data_dir}/processed/test.parquet")
y_test = pd.read_parquet(f"{data_dir}/raw/test_labels.parquet").values.flatten()
y_test = remap_labels(y_test)
y_test = np.array([x.lower() for x in y_test])

### Contact train and val

In [34]:
X_train = pd.concat([X_train, X_validation], ignore_index=True)
y_train = np.array([x.lower() for x in itertools.chain(y_train, y_validation)])

### Train Voting Classifier using RFC and ETC

In [62]:
# n_estimators=300 gives a slightly better result (0.1%), but triples the fit time
voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, random_state=13, n_jobs=-1)),
        ('et', ExtraTreesClassifier(n_estimators=100, random_state=13, n_jobs=-1))
    ],
    voting='soft'
)

start = datetime.now()
print(f'Started at {start}')

# label encoder
le = LabelEncoder()
le.fit(y_train)
y_train_enc = le.transform(y_train)

# train model
voting_clf.fit(X_train, y_train_enc)

print(f'Finished at {datetime.now()}, took {datetime.now() - start} seconds')

Started at 2025-05-16 13:28:43.227080
Finished at 2025-05-16 13:28:44.696112, took 0:00:01.469071 seconds


In [63]:
# Make individual (trained) estimators available
rf_clf = voting_clf.estimators_[0]
et_clf = voting_clf.estimators_[1]

## Make predictions

In [69]:
def predicted_labels(y_pred_proba, encoder):
    y_pred_int = y_pred_proba.argmax(axis=1)
    return encoder.inverse_transform(y_pred_int)

### Predict: RFC

In [75]:
pred_rfs_prob = predicted_labels(rf_clf.predict_proba(X_test), le)
print("F1 (weighted):", f1_score(y_test, pred_rfs_prob, average="weighted"))

F1 (weighted): 0.6407593191122604


### Predict: ETC

In [76]:
pred_etc_prob = predicted_labels(et_clf.predict_proba(X_test), le)
print("F1 (weighted):", f1_score(y_test, pred_etc_prob, average="weighted"))

F1 (weighted): 0.802095238095238


### Predict: Voting Classifier (RFC + ETC)

In [77]:
pred_voting_prob = predicted_labels(voting_clf.predict_proba(X_test), le)
print("F1 (weighted):", f1_score(y_test, pred_voting_prob, average="weighted"))

F1 (weighted): 0.7949890109890109


In [59]:
predicted_rfc_proba = rf_clf.predict_proba(X_test)

In [83]:
print(classification_report(y_test, pred_etc_prob, zero_division=0))

                        precision    recall  f1-score   support

                   age       0.50      0.50      0.50         2
           case_status       0.75      1.00      0.86         3
   confirmation_method       0.00      0.00      0.00         1
       contact_setting       0.00      0.00      0.00         0
                  date       0.88      1.00      0.93         7
           demographic       0.00      0.00      0.00         1
                gender       1.00      1.00      1.00         3
     genomics_metadata       0.00      0.00      0.00         1
                    id       1.00      1.00      1.00         5
              location       0.67      0.67      0.67         6
       medical_boolean       0.71      1.00      0.83         5
               outcome       1.00      0.50      0.67         2
pre_existing_condition       1.00      0.33      0.50         3
                source       1.00      1.00      1.00         1
              symptoms       1.00      

### Features

In [84]:
from matplotlib import pyplot as plt

# 1) Get importances
rf_imp = rf_clf.feature_importances_
et_imp = et_clf.feature_importances_

# 2) Average them (simple mean)
avg_imp = (rf_imp + et_imp) / 2.0

# 3) Create a DataFrame for easy sorting/plotting
feat_names = X_train.columns  # or whatever your feature names are
imp_df = pd.DataFrame({
    'feature': feat_names,
    'rf': rf_imp,
    'et': et_imp,
    'avg': avg_imp
}).sort_values('avg', ascending=False)

# 4) View top 10
print(imp_df.head(10))

# 5) Plot
plt.figure(figsize=(8, 6))
plt.barh(imp_df['feature'].head(10)[::-1],
         imp_df['avg'].head(10)[::-1])
plt.xlabel("Mean Feature Importance")
plt.title("Top 10 Features (averaged RF + ET)")
plt.tight_layout()
plt.show()

             feature        rf        et       avg
1164  frac_textcells  0.005081  0.009030  0.007056
1167  avg_text_cells  0.007888  0.003500  0.005694
10     n_[1]-agg-any  0.001235  0.009476  0.005356
7      n_[0]-agg-sum  0.009030  0.001596  0.005313
741    n_[-]-agg-all  0.001489  0.008716  0.005102
52    n_[5]-agg-mean  0.005991  0.004213  0.005102
51     n_[5]-agg-all  0.002552  0.007313  0.004933
740    n_[-]-agg-any  0.002289  0.006793  0.004541
21     n_[2]-agg-all  0.003458  0.005496  0.004477
31     n_[3]-agg-all  0.002751  0.006090  0.004420


<Figure size 800x600 with 1 Axes>

###

### Predictions custom

In [35]:
data = pd.Series(
    [
        ["2024-02-30", "2020-03-01", "1982-12-30"],
        ["104805", "330956", "345609"],
        ["Male", "Female"],
        ["men", "women"],

    ],
    name="values"
)

In [36]:
data

0    [2024-02-30, 2020-03-01, 1982-12-30]
1                [104805, 330956, 345609]
2                          [Male, Female]
3                            [men, women]
Name: values, dtype: object

In [38]:
from sherlock.features.preprocessing import extract_features

extract_features(
    "../check.csv",
    data
)
feature_vectors = pd.read_csv("../check.csv", dtype=np.float32)

Extracting Features: 100%|██████████| 4/4 [00:00<00:00, 489.47it/s]

Exporting 1588 column features


In [39]:
feature_vectors

,n_[0]-agg-any,n_[0]-agg-all,n_[0]-agg-mean,n_[0]-agg-var,n_[0]-agg-min,n_[0]-agg-max,n_[0]-agg-median,n_[0]-agg-sum,n_[0]-agg-kurtosis,n_[0]-agg-skewness,...,par_vec_390,par_vec_391,par_vec_392,par_vec_393,par_vec_394,par_vec_395,par_vec_396,par_vec_397,par_vec_398,par_vec_399
0,1.0,1.0,2.666667,1.555556,1.0,4.0,3.0,8.0,-1.5,-0.381802,...,0.000482,-0.000699,0.000464,0.000695,0.000377,0.000414,-0.000623,-0.000379,-0.000771,-0.000442
1,1.0,1.0,1.333333,0.222222,1.0,2.0,1.0,4.0,-1.5,0.707107,...,0.000800,0.000959,0.000994,0.000532,0.000768,0.000870,-0.000838,-0.000695,0.000956,0.000615
2,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,-3.0,0.000000,...,-0.024608,-0.009890,-0.016155,0.000928,-0.008309,-0.015509,-0.020144,-0.037095,-0.001885,-0.040874
3,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,-3.0,0.000000,...,-0.072729,-0.022693,-0.110678,0.089974,-0.099107,-0.090429,-0.001951,-0.008750,0.021203,-0.082619


In [44]:
train_columns_means = pd.DataFrame(feature_vectors.mean()).transpose()
feature_vectors.fillna(train_columns_means.iloc[0], inplace=True)

In [45]:
model = rf_clf
#predicted_labels = model.predict(data)

# 1) Ensemble predictions on training set
y_pred_ensemble = voting_clf.predict(feature_vectors)

print(y_pred_ensemble)

# 2) RandomForest‐only predictions on training set
#y_pred_rf = rf_clf.predict(X_train)

['date_report' 'id' 'outcome' 'pre_existing_condition']
